# D-01. 미등록 바닥 판별 — 특성 MLP 임베딩, 재질 하나 빼기 8회

- 할 일: 재질 하나를 학습에서 통째로 뺀 모델이 그 재질 구간을 "미등록"으로 판별하는지 측정
- 입력: B의 특성 MLP 임베딩(32차원)과 logits — `emb_mlp_{빠진재질}_{fold}.npy`, `logits_mlp_{빠진재질}_{fold}.npy`
  - 받은 zip(`ML-imu_embeddings…zip`)을 내 드라이브 맨 위에 올려 두면 0번 셀이 풀어서 쓴다
- 분할: `splits/folds.json`의 `material_out` 147개 (재질 8종 × 남은 녹화 하나씩 테스트)
- 비교 방법 5종 (모두 점수가 클수록 미등록)
  - 프로토타입 거리(제안), Mahalanobis(Lee et al., 2018), KNN 거리(Sun et al., 2022) — 임베딩
  - softmax 최대 확률(Hendrycks & Gimpel, 2017), Energy(Liu et al., 2020) — logits
- τ: 학습 녹화 안의 검증 녹화(`splits.inner_val_recordings`)에서 알려진 재질 95%가 통과하는 거리
- 지표: AUROC, FPR@95%TPR (fold별 계산 → 빠진 재질별 평균), near/far 구분 보고

실행: Colab에서 위에서부터 차례로 (CPU로 충분)

## 0. 환경 준비

In [ ]:
import os
import sys
from pathlib import Path

BRANCH = "D/ood-mlp"   # main에 merge된 뒤에는 "main"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/ML-imu").exists():
        !git clone -q https://github.com/ekgP525/ML-imu.git /content/ML-imu
    os.chdir("/content/ML-imu")
    !git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
    !pip install -q -r requirements.txt
    !python data/download.py

ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path += [str(ROOT / "src"), str(ROOT / "notebooks" / "D_ood")]
print(ROOT)

In [ ]:
# 임베딩 위치: 받은 zip을 내 드라이브 맨 위(MyDrive/)에 올려 두면 Colab 안에 풀어서 쓴다
# (드라이브에 풀린 폴더 MyDrive/ML-imu_embeddings가 있으면 그 폴더를 바로 쓴다)
EMB_ZIP = "ML-imu_embeddings*.zip"

if IN_COLAB:
    import zipfile
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    OUT = MYDRIVE / "ML-imu_D"
    EMB = MYDRIVE / "ML-imu_embeddings"
    if not EMB.exists():
        EMB = ROOT / "embeddings" / "mlp"
        if not any(EMB.glob("emb_mlp_*.npy")):
            zips = sorted(MYDRIVE.glob(EMB_ZIP))
            assert zips, f"MyDrive/에 {EMB_ZIP} 파일을 올려 주세요"
            EMB.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zips[-1]) as zf:
                for info in zf.infolist():
                    if not info.is_dir():
                        (EMB / Path(info.filename).name).write_bytes(zf.read(info))
            print("압축 해제:", zips[-1].name)
else:
    EMB = ROOT / "embeddings" / "mlp"
    OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

n_emb = len(list(EMB.glob("emb_mlp_*.npy")))
print(EMB, n_emb, "개 임베딩 (기대값 168 = 147 + 21)")

## 1. 데이터와 분할

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # 일부 환경의 NumPy matmul 경고 (값에는 영향 없음)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recordings import load_dataset
from splits import load_material_folds
from evaluate import log_result
from ood import Prototype, confident_mistakes, run_heldout, summarize, DIFFICULTY

np.random.seed(42)
X, y, rec = load_dataset("data")
folds = load_material_folds()
print(X.shape, len(np.unique(rec)), "개 녹화 |", len(folds), "개 split")
print(pd.Series([f["held_out"] for f in folds]).value_counts().sort_index().to_dict())

## 2. 재질 하나 빼기 147 split 실행

In [ ]:
MODEL = "mlp"

def load(held_out, fold):
    tag = f"{MODEL}_{held_out}_{fold}"
    return np.load(EMB / f"emb_{tag}.npy"), np.load(EMB / f"logits_{tag}.npy")

z0, lg0 = load(folds[0]["held_out"], folds[0]["fold"])
assert len(z0) == len(y), (z0.shape, y.shape)
print("emb", z0.shape, "| logits", lg0.shape)

df = run_heldout(load, y, rec, model=MODEL, folds=folds)
df.to_csv(OUT / f"D_heldout_{MODEL}_folds.csv", index=False)
print(len(df), "행 저장 →", OUT / f"D_heldout_{MODEL}_folds.csv")

## 3. 결과 표

In [ ]:
METHODS = ["proto", "maha", "knn", "msp", "energy"]
order = sorted(df.heldout.unique(), key=lambda m: ({"far": 0, "mid": 1, "near": 2}[DIFFICULTY.get(m, "mid")], m))

auroc_tab = df.pivot_table(index="heldout", columns="method", values="auroc").loc[order, METHODS]
fpr_tab = df.pivot_table(index="heldout", columns="method", values="fpr95").loc[order, METHODS]
auroc_tab.loc["평균"] = auroc_tab.mean()
fpr_tab.loc["평균"] = fpr_tab.mean()
print("AUROC (빠진 재질별 fold 평균, 클수록 좋음)")
display(auroc_tab.round(3))
print("FPR@95%TPR (작을수록 좋음)")
display(fpr_tab.round(3))

In [ ]:
by_material, by_difficulty = summarize(df)
print("near / mid / far 별 평균")
display(by_difficulty.unstack("difficulty").round(3))
print("fold 간 표준편차 (AUROC)")
display(df.pivot_table(index="heldout", columns="method", values="auroc", aggfunc="std").loc[order, METHODS].round(3))

## 4. τ 판정 (제안 방법)

- `known_pass`: 알려진 재질 테스트 구간 중 τ 안으로 들어온 비율 (목표 0.95)
- `unknown_reject`: 빠진 재질 구간 중 미등록으로 판정한 비율

In [ ]:
tau_tab = (df[df.method == "proto"]
           .groupby("heldout")[["tau", "known_pass", "unknown_reject"]].mean()
           .loc[order])
tau_tab.loc["평균"] = tau_tab.mean()
display(tau_tab.round(3))

## 5. 그림: 빠진 재질별 AUROC

In [ ]:
ax = auroc_tab.drop("평균").plot.bar(figsize=(10, 4), width=0.8)
ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_ylabel("AUROC"); ax.set_xlabel("held-out material (far → near)")
ax.set_ylim(0, 1.02); ax.legend(ncol=5, loc="lower left")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(OUT / f"D_auroc_{MODEL}.png", dpi=150)
plt.show()

## 6. 확신하며 틀린 사례

빠진 재질 구간 중 프로토타입 거리가 가장 짧은(가장 확신 있게 기존 재질로 받아들인) 구간. 재질마다 fold 0 기준

In [ ]:
rows = []
for m in order:
    split = next(f for f in folds if f["held_out"] == m and f["fold"] == 0)
    z, _ = load(m, 0)
    train = np.isin(rec, split["train"])
    proto = Prototype().fit(z[train], y[train])
    idx_unknown = np.where(np.isin(rec, split["unknown"]))[0]
    cm = confident_mistakes(proto, z, y, idx_unknown, n=5)
    cm["recording"] = rec[cm["index"]]
    rows.append(cm)
mistakes = pd.concat(rows, ignore_index=True)
mistakes.to_csv(OUT / f"D_confident_mistakes_{MODEL}.csv", index=False)
display(mistakes.round(3))
print("가장 많이 받아들여진 기존 재질:")
display(mistakes.groupby("true")["pred"].agg(lambda s: s.value_counts().to_dict()))

## 7. results.csv 기록

방법 × 빠진 재질별 fold 평균을 한 줄씩 기록 (`fold="mean"`). fold별 원본은 `D_heldout_mlp_folds.csv`

In [ ]:
summary = df.groupby(["method", "heldout"])[["auroc", "fpr95"]].mean().reset_index()
for _, r in summary.iterrows():
    log_result("D", f"{MODEL}+{r.method}", {"auroc": r.auroc, "fpr95": r.fpr95},
               setting=f"heldout={r.heldout}", fold="mean",
               note=f"{DIFFICULTY.get(r.heldout, 'mid')}, material_out {(df.heldout == r.heldout).sum() // df.method.nunique()} folds")

import shutil
shutil.copy(ROOT / "results" / "results.csv", OUT / "results_with_D.csv")
print(len(summary), "줄 기록 → results/results.csv (드라이브 사본:", OUT / "results_with_D.csv", ")")

## 메모할 것

- 가장 좋은 방법과 평균 AUROC / FPR@95:
- near(concrete·tiled·hard_tiles) vs far(carpet) 차이:
- τ: 알려진 재질 통과율이 0.95에 가까운지 (검증 녹화도 MLP 학습에 들어가 있어 τ가 빡빡할 수 있음)
- 확신하며 틀린 사례: 어떤 재질로 받아들였는지
- 드라이브 산출물: `ML-imu_D/` 폴더의 csv·png